In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [3]:
# Get the base data of the csv files downloaded from our datasets
base = Path("data/FoodData_Central_foundation_food_csv_2026-10-29")
survey = Path("data/FoodData_Central_survey_food_csv_2024-10-31")
nutrient_base = Path("data/FoodData_Central_foundation_food_csv_2026-10-29")
nutrient = Path("data/FoodData_Central_Supporting_Data_csv_2021-04-28")
nutrient_survey = Path("data/FoodData_Central_survey_food_csv_2024-10-31")

#food = pd.read_csv("data/FoodData_Central_foundation_food_csv_2021-04-28/food.csv")
#food_survey = pd.read_csv("data/FoodData_Central_survey_food_csv_2020-10-30/food.csv")
#nutrient = pd.read_csv("data/FoodData_Central_Supporting_Data_csv_2021-04-28/nutrient.csv")
#food_nutrient = pd.read_csv("data/FoodData_Central_foundation_food_csv_2021-04-28/food_nutrient.csv")
#food_nutrient_survey = pd.read_csv("data/FoodData_Central_survey_food_csv_2020-10-30/food_nutrient.csv")

# Get the CSV files and read them using pandas
food = pd.read_csv(base/"food.csv")
food_survey = pd.read_csv(survey/"food.csv")
nutrients = pd.read_csv(nutrient/"nutrient.csv")
food_nutrients = pd.read_csv(nutrient_base/"food_nutrient.csv")
food_nutrients_survey = pd.read_csv(nutrient_survey/"food_nutrient.csv")

# Print out the length of each of the following csv files to get insights
print(len(food), len(food_survey), len(nutrients), len(food_nutrients), len(food_nutrients_survey))

C:\Users\Max\AppData\Local\Temp\ipykernel_2128\705232962.py:18: DtypeWarning: Columns (0: footnote) have mixed types. Specify dtype option on import or set low_memory=False.
  food_nutrients = pd.read_csv(nutrient_base/"food_nutrient.csv")


101532 5432 474 189236 353015


In [4]:
food.columns

Index(['fdc_id', 'data_type', 'description', 'food_category_id',
       'publication_date'],
      dtype='str')

In [5]:
food_survey.columns

Index(['fdc_id', 'data_type', 'description', 'food_category_id',
       'publication_date'],
      dtype='str')

In [6]:
food_nutrients.columns

Index(['id', 'fdc_id', 'nutrient_id', 'amount', 'data_points', 'derivation_id',
       'min', 'max', 'median', 'footnote', 'min_year_acquired'],
      dtype='str')

In [7]:
food_nutrients_survey.columns

Index(['id', 'fdc_id', 'nutrient_id', 'amount', 'data_points', 'derivation_id',
       'min', 'max', 'median', 'footnote', 'min_year_acquired'],
      dtype='str')

In [8]:
# merge food and food_survey as they are all have the same column heads and fit together
food = pd.concat([food, food_survey], ignore_index=True).dropna(subset=["description"])
food["description"] = food["description"].str.lower()
print(f"Combined food rows: {len(food)}")

food_nutrients = pd.concat([food_nutrients, food_nutrients_survey], ignore_index=True)
food_nutrients["nutrient_name"] = food_nutrients["nutrient_id"].map(nutrients.set_index("id")["name"]).str.lower()
print(f"Combined food nutrient rows: {len(food_nutrients)}")

Combined food rows: 106956
Combined food nutrient rows: 542251


In [9]:
#food.columns

In [10]:
#food_nutrients.columns
food_nutrients.head()

,id,fdc_id,nutrient_id,amount,data_points,derivation_id,min,max,median,footnote,min_year_acquired,nutrient_name
0,2201847,319877,1051,56.30,1.0,1.0,NaN,NaN,NaN,NaN,NaN,water
1,2201845,319877,1002,1.28,1.0,1.0,NaN,NaN,NaN,NaN,NaN,nitrogen
2,2201846,319877,1004,19.00,1.0,1.0,NaN,NaN,NaN,NaN,NaN,total lipid (fat)
3,2201844,319877,1007,1.98,1.0,1.0,NaN,NaN,NaN,NaN,NaN,ash
4,2201852,319878,1091,188.00,1.0,1.0,NaN,NaN,NaN,NaN,NaN,"phosphorus, p"


In [13]:
# find unique dataset
unique_description = food["description"].unique()
print(len(unique_description))

17159


We will work with this unique description to model our project. We weill try to model at about ~17200 data classes

In [14]:
unique_description[:10]

<StringArray>
['hummus, sabra classic',                'hummus',         'hummus, other',
    'hummus - nfy12140o',    'hummus - nfy12140p',    'hummus - nfy12140q',
    'hummus - nfy12140r',    'hummus - nfy12140s',    'hummus - nfy12140f',
    'hummus - nfy12140g']
Length: 10, dtype: str

In [15]:
unique_description[-10:]

<StringArray>
[                'celery, cooked, as ingredient',
 'dark green vegetables as ingredient in omelet',
              'tomatoes as ingredient in omelet',
      'other vegetables as ingredient in omelet',
               'mirepoix, cooked, as ingredient',
             'vegetables as ingredient in curry',
             'vegetables as ingredient in soups',
             'vegetables as ingredient in stews',
             'sauce as ingredient in hamburgers',
          'industrial oil as ingredient in food']
Length: 10, dtype: str

In [16]:
# Find random indexes of food to explore
import random
random_number = random.randint(0, len(unique_description)-10)
unique_description[random_number:random_number+10]

<StringArray>
['sauce, pasta, ragu old world, bottled spaghetti sauce, 26 oz container (al1) - nfy0905jj',
 'sauce, pasta, ragu old world, bottled spaghetti sauce, 26 oz container (al1) - nfy0905ji',
        'minerals, sauce, pasta, ragu old world, bottled spaghetti sauce (al1) - nfy0904rv',
      'proximates, sauce, pasta, ragu old world, bottled spaghetti sauce (al1) - nfy0904rx',
                    'sauce, pasta, ragu old world, bottled spaghetti sauce (ca2) - cy0907i',
 'sauce, pasta, ragu old world, bottled spaghetti sauce, 26 oz container (ca2) - nfy0905kk',
        'minerals, sauce, pasta, ragu old world, bottled spaghetti sauce (ca2) - nfy0904s7',
      'proximates, sauce, pasta, ragu old world, bottled spaghetti sauce (ca2) - nfy0904s9',
                    'sauce, pasta, ragu old world, bottled spaghetti sauce (nc1) - cy0907j',
 'sauce, pasta, ragu old world, bottled spaghetti sauce, 26 oz container (nc1) - nfy0905k5']
Length: 10, dtype: str

### Food Categories

Let us dive deep into food categories for our datasets and explore them and how they relate to the nutrient values

In [23]:
food.head()

,fdc_id,data_type,description,food_category_id,publication_date
0,319874,sample_food,"hummus, sabra classic",16.0,2019-04-01
1,319875,market_acquisition,"hummus, sabra classic",16.0,2019-04-01
2,319876,market_acquisition,"hummus, sabra classic",16.0,2019-04-01
3,319877,sub_sample_food,hummus,16.0,2019-04-01
4,319878,sub_sample_food,hummus,16.0,2019-04-01


In [24]:
unique_food_category = food["food_category_id"].unique()
print(len(unique_food_category))

192


In [25]:
# Checking out which are these 192 unique food categories sets we have...
food["food_category_id"].value_counts()

food_category_id
11.0      17641
9.0       12448
1.0       10970
20.0       9594
16.0       8744
          ...  
7804.0        2
9602.0        1
6006.0        1
7704.0        1
9204.0        1
Name: count, Length: 191, dtype: int64

In [26]:
# Get food categories
food_cats = pd.read_csv("data/FoodData_Central_foundation_food_csv_2026-10-29/food_category.csv")
food_cats

,id,code,description
0,1,100,Dairy and Egg Products
1,2,200,Spices and Herbs
2,3,300,Baby Foods
3,4,400,Fats and Oils
4,5,500,Poultry Products
5,6,600,"Soups, Sauces, and Gravies"
6,7,700,Sausages and Luncheon Meats
7,8,800,Breakfast Cereals
8,9,900,Fruits and Fruit Juices
9,10,1000,Pork Products


Foods we want. We will choose 10 for this file here that we are sure they are on our data and are whole

In [29]:
ten_whole_foods = ['apple',
 'banana',
 'beef', # steak etc
 'blueberries',
 'carrots',
 'chicken_wings',
 'egg', # whole egg
 'honey',
 'mushrooms',
 'strawberries']
ten_whole_foods

['apple',
 'banana',
 'beef',
 'blueberries',
 'carrots',
 'chicken_wings',
 'egg',
 'honey',
 'mushrooms',
 'strawberries']

In [30]:
food.head()

,fdc_id,data_type,description,food_category_id,publication_date
0,319874,sample_food,"hummus, sabra classic",16.0,2019-04-01
1,319875,market_acquisition,"hummus, sabra classic",16.0,2019-04-01
2,319876,market_acquisition,"hummus, sabra classic",16.0,2019-04-01
3,319877,sub_sample_food,hummus,16.0,2019-04-01
4,319878,sub_sample_food,hummus,16.0,2019-04-01


In [48]:
# Foundation food is the ground truth for a certain type of food, excludes some details about the food
# E.g. the data_type foundation_food for Chicken will the the original unique ID for chicken
foundation_food = food[(food["data_type"] == "foundation_food") | (food["data_type"] == "survey_fndds_food")]
#foundation_food
len(foundation_food)

5930

In [49]:
foundation_food[:10]

,fdc_id,data_type,description,food_category_id,publication_date
651,321358,foundation_food,"hummus, commercial",16.0,2019-04-01
652,321359,foundation_food,"milk, reduced fat, fluid, 2% milkfat, with add...",1.0,2019-04-01
653,321360,foundation_food,"tomatoes, grape, raw",11.0,2019-04-01
798,321505,foundation_food,"salt, table, iodized",2.0,2019-04-01
904,321611,foundation_food,"beans, snap, green, canned, regular pack, drai...",11.0,2019-04-01
1193,321900,foundation_food,"broccoli, raw",11.0,2019-04-01
1521,322228,foundation_food,"milk, lowfat, fluid, 1% milkfat, with added vi...",1.0,2019-04-01
1852,322559,foundation_food,"milk, nonfat, fluid, with added vitamin a and ...",1.0,2019-04-01
2185,322892,foundation_food,"milk, whole, 3.25% milkfat, with added vitamin d",1.0,2019-04-01
2414,323121,foundation_food,"frankfurter, beef, unheated",7.0,2019-04-01


In [50]:
# This will find all foods starting with blue e.g blueberries... etc
foundation_food[foundation_food["description"].str.contains("blue")]

,fdc_id,data_type,description,food_category_id,publication_date
42186,2263889,foundation_food,"blueberries, raw",9.0,2022-04-28
43441,2346411,foundation_food,"blueberries, raw",9.0,2022-10-28
60660,2684446,foundation_food,"crustaceans, crab, blue swimming, lump, pasteu...",15.0,2024-04-18
101854,2705705,survey_fndds_food,"cheese, blue or roquefort",1602.0,2022-10-28
104147,2707998,survey_fndds_food,"pie, blueberry",5502.0,2022-10-28
105347,2709198,survey_fndds_food,"blueberries, dried",6016.0,2022-10-28
105424,2709275,survey_fndds_food,"blueberries, raw",6011.0,2022-10-28
105426,2709277,survey_fndds_food,"blueberries, frozen",6011.0,2022-10-28
105427,2709278,survey_fndds_food,blueberry pie filling,8806.0,2022-10-28
105472,2709323,survey_fndds_food,blueberry juice,7006.0,2022-10-28


In [53]:
#print(foundation_food.columns)

In [52]:
foundation_foods = foundation_food["description"]
foundation_foods[20:40]

4153               peanut butter, smooth style, with salt
4329                             cheese, parmesan, grated
4491    cheese, pasteurized process, american, vitamin...
4580    grapefruit juice, white, canned or bottled, un...
4723                                 peaches, yellow, raw
4817    seeds, sunflower seed kernels, dry roasted, wi...
4951      sausage, italian, pork, mild, cooked, pan-fried
5164                  bread, white, commercially prepared
5285          sausage, turkey, breakfast links, mild, raw
5428                                        cheese, swiss
5489    kale, frozen, cooked, boiled, drained, without...
5751    carrots, frozen, unprepared (includes foods fo...
5991                            mustard, prepared, yellow
6198                                figs, dried, uncooked
6339                                kiwifruit, green, raw
6491                              melons, cantaloupe, raw
6650                                      nectarines, raw
6794    orange

In [54]:
# Found a list of the foundation foods we're going to start with!
foundation_foods_list = list(foundation_foods)
for food in foundation_foods_list:
    if "blue" in food:
        print(food)

blueberries, raw
blueberries, raw
crustaceans, crab, blue swimming, lump, pasteurized, refrigerated
cheese, blue or roquefort
pie, blueberry
blueberries, dried
blueberries, raw
blueberries, frozen
blueberry pie filling
blueberry juice
blue or roquefort cheese dressing
blue or roquefort cheese dressing, light
blue or roquefort cheese dressing, fat free
blueberry syrup


In [59]:
#food.loc[(food["description"].str.contains("chicken", case=False)) & (food["description"].str.contains("drumstick", case=False))][-10:]
# Let us search for chicken in our food datasets list
for food in foundation_foods_list:
    if "chicken" in food.lower():
        print(food)

chicken, broilers or fryers, drumstick, meat only, cooked, braised
chicken, broiler or fryers, breast, skinless, boneless, meat only, cooked, braised
chicken, ground, with additives, raw
chicken, breast, boneless, skinless, raw
chicken, thigh, boneless, skinless, raw
chicken, drumstick, meat and skin, raw
chicken, thigh, meat and skin, raw
chicken, wing, meat and skin, raw
chicken, breast, meat and skin, raw
lunchmeat, chicken breast, sliced
mock chicken legs
chicken, ns as to part and cooking method, ns as to skin eaten
chicken, ns as to part and cooking method, skin eaten
chicken, ns as to part and cooking method, skin not eaten
chicken, ns as to part, baked, broiled, or roasted, ns as to skin eaten
chicken, ns as to part, baked, broiled, or roasted, skin eaten
chicken, ns as to part, baked, broiled, or roasted, skin not eaten
chicken, ns as to part, rotisserie, ns as to skin eaten
chicken, ns as to part, rotisserie, skin eaten
chicken, ns as to part, rotisserie, skin not eaten
chick

In [61]:
# Try to find chicken wing ID or the dataset....
# The loc is used to access the location of something in a dataset in our case the csv file...
chicken_wing_id = int(foundation_food.loc[foundation_food["description"].str.contains("Chicken", case=False)].iloc[0]["fdc_id"])
chicken_wing_id

331897

In [63]:
# Find the nutrients in food_nutrients for the chicken wing we are looking at.
food_nutrients[food_nutrients["fdc_id"] == chicken_wing_id]

,id,fdc_id,nutrient_id,amount,data_points,derivation_id,min,max,median,footnote,min_year_acquired,nutrient_name
41714,2259068,331897,1303,0.003,5.0,1.0,0.002,0.004,0.003,NaN,2010.0,tfa 16:1 t
41715,2259065,331897,1280,0.008,5.0,1.0,0.008,0.009,0.008,NaN,2010.0,pufa 22:5 n-3 (dpa)
41716,2259076,331897,1404,0.045,5.0,1.0,0.035,0.059,0.042,NaN,2010.0,"pufa 18:3 n-3 c,c,c (ala)"
41717,2259059,331897,1261,0.002,5.0,1.0,0.001,0.003,0.002,NaN,2010.0,sfa 8:0
41718,2259106,331897,1109,0.170,1.0,1.0,NaN,NaN,0.170,NaN,2010.0,vitamin e (alpha-tocopherol)
...,...,...,...,...,...,...,...,...,...,...,...,...
41806,2259112,331897,1167,5.050,5.0,1.0,4.890,5.240,5.050,NaN,2010.0,niacin
41807,2259074,331897,1329,0.021,NaN,4.0,NaN,NaN,NaN,NaN,NaN,"fatty acids, total trans-monoenoic"
41808,2259138,331897,1330,0.008,NaN,4.0,NaN,NaN,NaN,NaN,NaN,"fatty acids, total trans-dienoic"
41809,13338545,331897,2047,149.000,NaN,1.0,NaN,NaN,NaN,NaN,NaN,energy (atwater general factors)


### **Get Protein, fat, carb IDs**

In [64]:
food_nutrients.head()

,id,fdc_id,nutrient_id,amount,data_points,derivation_id,min,max,median,footnote,min_year_acquired,nutrient_name
0,2201847,319877,1051,56.30,1.0,1.0,NaN,NaN,NaN,NaN,NaN,water
1,2201845,319877,1002,1.28,1.0,1.0,NaN,NaN,NaN,NaN,NaN,nitrogen
2,2201846,319877,1004,19.00,1.0,1.0,NaN,NaN,NaN,NaN,NaN,total lipid (fat)
3,2201844,319877,1007,1.98,1.0,1.0,NaN,NaN,NaN,NaN,NaN,ash
4,2201852,319878,1091,188.00,1.0,1.0,NaN,NaN,NaN,NaN,NaN,"phosphorus, p"


In [76]:
nutrients[(nutrients["name"].str.contains("protein", case=False)) | \
         (nutrients["name"].str.contains("carbohydrate", case=False)) | \
         (nutrients["name"].str.contains("fat", case=False))]

,id,name,unit_name,nutrient_nbr,rank
4,1003,Protein,G,203.0,600.0
5,1004,Total lipid (fat),G,204.0,800.0
6,1005,"Carbohydrate, by difference",G,205.0,1110.0
50,1049,"Solids, non-fat",G,253.0,999999.0
51,1050,"Carbohydrate, by summation",G,205.2,1120.0
54,1053,Adjusted Protein,G,257.0,700.0
73,1072,"Carbohydrate, other",G,284.0,NaN
86,1085,Total fat (NLEA),G,298.0,900.0
258,1257,"Fatty acids, total trans",G,605.0,15400.0
259,1258,"Fatty acids, total saturated",G,606.0,9700.0


aLL these are in 100 g

In [78]:
target_nutrients = nutrients[nutrients["name"].isin(["Protein", "Total lipid (fat)", "Carbohydrate, by difference"])]
target_nutrients

,id,name,unit_name,nutrient_nbr,rank
4,1003,Protein,G,203.0,600.0
5,1004,Total lipid (fat),G,204.0,800.0
6,1005,"Carbohydrate, by difference",G,205.0,1110.0


In [79]:
target_nutrient_dict = {1003: "protein",
    1004: "fat",
    1005: "carbohydrate"
}

### Get target food protein, fat, carbohydrates

In [80]:
list(target_nutrient_dict.keys())

[1003, 1004, 1005]

In [81]:
food_nutrients

,id,fdc_id,nutrient_id,amount,data_points,derivation_id,min,max,median,footnote,min_year_acquired,nutrient_name
0,2201847,319877,1051,56.30,1.0,1.0,NaN,NaN,NaN,NaN,NaN,water
1,2201845,319877,1002,1.28,1.0,1.0,NaN,NaN,NaN,NaN,NaN,nitrogen
2,2201846,319877,1004,19.00,1.0,1.0,NaN,NaN,NaN,NaN,NaN,total lipid (fat)
3,2201844,319877,1007,1.98,1.0,1.0,NaN,NaN,NaN,NaN,NaN,ash
4,2201852,319878,1091,188.00,1.0,1.0,NaN,NaN,NaN,NaN,NaN,"phosphorus, p"
...,...,...,...,...,...,...,...,...,...,...,...,...
542246,34489112,2710814,208,892.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
542247,34489151,2710814,601,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
542248,34489135,2710814,337,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
542249,34489121,2710814,304,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [86]:
food_nutrients[(food_nutrients["nutrient_id"].isin(list(target_nutrient_dict.keys())))]

,id,fdc_id,nutrient_id,amount,data_points,derivation_id,min,max,median,footnote,min_year_acquired,nutrient_name
2,2201846,319877,1004,19.00,1.0,1.0,NaN,NaN,NaN,NaN,NaN,total lipid (fat)
16,2201859,319882,1004,18.70,1.0,1.0,NaN,NaN,NaN,NaN,NaN,total lipid (fat)
28,2201873,319892,1004,16.60,1.0,1.0,NaN,NaN,NaN,NaN,NaN,total lipid (fat)
43,2201886,319899,1004,19.10,1.0,1.0,NaN,NaN,NaN,NaN,NaN,total lipid (fat)
97,2201942,319908,1004,18.20,1.0,1.0,NaN,NaN,NaN,NaN,NaN,total lipid (fat)
...,...,...,...,...,...,...,...,...,...,...,...,...
189132,35279723,2792643,1004,46.56,1.0,1.0,NaN,NaN,NaN,NaN,NaN,total lipid (fat)
189153,35279744,2792664,1004,44.35,1.0,1.0,NaN,NaN,NaN,NaN,NaN,total lipid (fat)
189174,35279765,2792685,1004,46.35,1.0,1.0,NaN,NaN,NaN,NaN,NaN,total lipid (fat)
189195,35279786,2792706,1004,33.39,1.0,1.0,NaN,NaN,NaN,NaN,NaN,total lipid (fat)


In [88]:
food_nutrients.dtypes

id                     int64
fdc_id                 int64
nutrient_id            int64
amount               float64
data_points          float64
derivation_id        float64
min                  float64
max                  float64
median               float64
footnote              object
min_year_acquired    float64
nutrient_name            str
dtype: object

In [99]:
# Find nutrition for chicken_wing_id (protein, fat, carb)
food_nutrients[(food_nutrients["fdc_id"] == chicken_wing_id) & (food_nutrients["nutrient_id"].isin(list(target_nutrient_dict.keys())))]

,id,fdc_id,nutrient_id,amount,data_points,derivation_id,min,max,median,footnote,min_year_acquired,nutrient_name
41750,2259098,331897,1004,5.95,6.0,1.0,5.54,6.33,5.93,NaN,2010.0,total lipid (fat)
41782,2259079,331897,1003,23.90,NaN,49.0,23.00,24.60,24.10,NaN,NaN,protein
41793,2259099,331897,1005,0.00,NaN,49.0,NaN,NaN,NaN,NaN,NaN,"carbohydrate, by difference"


In [101]:
sorted(list(foundation_foods))

['abalone',
 'adobo, with noodles',
 'adobo, with rice',
 'agave liquid sweetener',
 'alaska pollock, raw',
 'alcoholic coffee drink',
 'alcoholic malt beverage',
 'alcoholic malt beverage, sweetened',
 'alfalfa sprouts, raw',
 'alfredo sauce',
 'alfredo sauce with added vegetables',
 'alfredo sauce with meat',
 'alfredo sauce with meat and added vegetables',
 'alfredo sauce with poultry',
 'alfredo sauce with poultry and added vegetables',
 'alfredo sauce with seafood',
 'alfredo sauce with seafood and added vegetables',
 'almond butter',
 'almond butter and jelly sandwich, on wheat bread',
 'almond butter and jelly sandwich, on white bread',
 'almond butter sandwich, on wheat bread',
 'almond butter sandwich, on white bread',
 'almond butter, creamy',
 'almond butter, lower sodium',
 'almond chicken',
 'almond milk, chocolate',
 'almond milk, nfs',
 'almond milk, sweetened',
 'almond milk, unsweetened',
 'almond milk, unsweetened, plain, refrigerated',
 'almond milk, unsweetened, pla

In [102]:
ten_whole_foods = ["chicken_wings",
    "apple",
    "banana",
    "beef", # steak, etc
    "carrots",
    "egg", # whole egg
    "strawberries",
    "blueberries",
    "mushrooms",
    "honey"
]

In [103]:
ten_whole_foods

['chicken_wings',
 'apple',
 'banana',
 'beef',
 'carrots',
 'egg',
 'strawberries',
 'blueberries',
 'mushrooms',
 'honey']

### Get ten whole foods `food_id`

In [104]:
# Get all food ids from foundation_food (honey and blueberries in another dataset)
target_whole_foods = ['apple', # removed chicken wings... can come back later...
 'banana',
 'beef',
 'blueberries',
 'carrots',
 'chicken',
 'egg',
 'honey',
 'strawberries',
 'mushrooms']

In [111]:
import re

# str.contains can search on regex - https://stackoverflow.com/a/17973255/7900723
pattern = "|".join([re.escape(food) for food in target_whole_foods])
pattern

'apple|banana|beef|blueberries|carrots|chicken|egg|honey|strawberries|mushrooms'

In [112]:
foundation_food[foundation_food["description"].str.contains(pattern, case=False)].sort_values(by=["description"])

,fdc_id,data_type,description,food_category_id,publication_date
102946,2706797,survey_fndds_food,almond chicken,3404.0,2022-10-28
103640,2707491,survey_fndds_food,"almonds, honey roasted",2804.0,2022-10-28
105468,2709319,survey_fndds_food,apple cider,7004.0,2022-10-28
106740,2710591,survey_fndds_food,"apple juice beverage, 40-50% juice, light",7204.0,2022-10-28
105469,2709320,survey_fndds_food,"apple juice, 100%",7004.0,2022-10-28
...,...,...,...,...,...
102813,2706664,survey_fndds_food,"venison or deer, noodles, and vegetables inclu...",3002.0,2022-10-28
102812,2706663,survey_fndds_food,"venison or deer, potatoes, and vegetables excl...",3002.0,2022-10-28
102811,2706662,survey_fndds_food,"venison or deer, potatoes, and vegetables incl...",3002.0,2022-10-28
103680,2707531,survey_fndds_food,"walnuts, excluding honey roasted",2804.0,2022-10-28


In [113]:
foundation_food[foundation_food["description"].str.contains("honey")]

,fdc_id,data_type,description,food_category_id,publication_date
20191,1105547,foundation_food,"apples, honeycrisp, with skin, raw",9.0,2020-10-30
20547,1750343,foundation_food,"apples, honeycrisp, with skin, raw",9.0,2020-10-30
63976,2710816,foundation_food,"melons, honeydew, raw",9.0,2024-10-31
103640,2707491,survey_fndds_food,"almonds, honey roasted",2804.0,2022-10-28
103647,2707498,survey_fndds_food,"cashews, honey roasted",2804.0,2022-10-28
103660,2707511,survey_fndds_food,"mixed nuts, honey roasted",2804.0,2022-10-28
103669,2707520,survey_fndds_food,"peanuts, honey roasted",2804.0,2022-10-28
103674,2707525,survey_fndds_food,"pecans, honey roasted",2804.0,2022-10-28
103680,2707531,survey_fndds_food,"walnuts, excluding honey roasted",2804.0,2022-10-28
103681,2707532,survey_fndds_food,"walnuts, honey roasted",2804.0,2022-10-28


In [114]:
foundation_food[foundation_food["description"].str.contains("blueberries")]

,fdc_id,data_type,description,food_category_id,publication_date
42186,2263889,foundation_food,"blueberries, raw",9.0,2022-04-28
43441,2346411,foundation_food,"blueberries, raw",9.0,2022-10-28
105347,2709198,survey_fndds_food,"blueberries, dried",6016.0,2022-10-28
105424,2709275,survey_fndds_food,"blueberries, raw",6011.0,2022-10-28
105426,2709277,survey_fndds_food,"blueberries, frozen",6011.0,2022-10-28


In [115]:
# Map foods to food_id (these have been filtered from larger quantities to smaller quantities)
# For example, if there were 5 kinds of apple, only one was chosen
whole_foods_id_map = {1750339: "apple", # red delicious
    1105314: "banana", # Bananas, ripe and slightly ripe, raw
    1102702: "blueberries", # blueberries, raw	
    746763: "beef", # t-bone steak 
    746764: "carrots", # frozen unprepared
    331897: "chicken_wings", # Chicken, broilers or fryers, drumstick, meat o...	
    329490: "egg", # Egg, whole, dried	
    1103956: "honey", # Honey
    1750347: "mushrooms", # Mushrooms, white button
    747448: "strawberries" # strawberries, raw
}

In [117]:
list(whole_foods_id_map.keys())

[1750339,
 1105314,
 1102702,
 746763,
 746764,
 331897,
 329490,
 1103956,
 1750347,
 747448]

In [119]:
# Find nutrition for eight whole foods
target_whole_foods_df = food_nutrients[(food_nutrients["fdc_id"].isin(list(whole_foods_id_map.keys()))) & \
    (food_nutrients["nutrient_id"].isin(list(target_nutrient_dict.keys())))][["fdc_id", "nutrient_id", "amount"]]
target_whole_foods_df

,fdc_id,nutrient_id,amount
34317,329490,1004,39.800000
34318,329490,1005,1.870000
34322,329490,1003,48.100000
41750,331897,1004,5.950000
41782,331897,1003,23.900000
41793,331897,1005,0.000000
71149,746763,1003,27.300000
71176,746763,1005,0.000000
71194,746763,1004,11.400000
71274,746764,1004,0.470000


In [120]:
# Pivot the table to how we want it
target_whole_foods_df = target_whole_foods_df.pivot_table("amount", "fdc_id", "nutrient_id")
target_whole_foods_df

nutrient_id,1003,1004,1005
fdc_id,,,
329490,48.100000,39.8000,1.870000
331897,23.900000,5.9500,0.000000
746763,27.300000,11.4000,0.000000
746764,0.810000,0.4700,7.920000
747448,0.640000,0.2200,7.630000
1105314,0.740000,0.2900,23.000000
1750339,0.187500,0.2125,14.781700
1750347,2.890625,0.3708,4.079375


In [121]:
len(whole_foods_id_map)

10

In [123]:
target_nutrient_dict

{1003: 'protein', 1004: 'fat', 1005: 'carbohydrate'}

In [124]:
# Rename columns
target_whole_foods_df.rename(columns=target_nutrient_dict, inplace=True)
target_whole_foods_df

nutrient_id,protein,fat,carbohydrate
fdc_id,,,
329490,48.100000,39.8000,1.870000
331897,23.900000,5.9500,0.000000
746763,27.300000,11.4000,0.000000
746764,0.810000,0.4700,7.920000
747448,0.640000,0.2200,7.630000
1105314,0.740000,0.2900,23.000000
1750339,0.187500,0.2125,14.781700
1750347,2.890625,0.3708,4.079375


All amounts are per 100g

### Export first 10 target food nutrition information

In [125]:
target_whole_foods_df.to_csv("target_ten_whole_food_nutrition_info.csv", index=False)

In [126]:
ten_whole_foods

['chicken_wings',
 'apple',
 'banana',
 'beef',
 'carrots',
 'egg',
 'strawberries',
 'blueberries',
 'mushrooms',
 'honey']